<a href="https://colab.research.google.com/github/Chalhotra/ViT-Token-Economy/blob/main/notebooks/01_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baselines: ViT-Tiny and DeiT-Tiny (ImageNet-100)

This notebook is intentionally thin: it calls into `src/` modules.

## Setup (Colab)

Run this cell to clone the repository. For private repos, you'll need a GitHub token with repo access.

In [ ]:
import os
import getpass

# For public repos, you can clone without a token:
# !git clone https://github.com/Chalhotra/ViT-Token-Economy.git

# For private repos, use a token (stored in Colab secrets or entered via getpass):
if 'GITHUB_TOKEN' in os.environ:
    token = os.environ['GITHUB_TOKEN']
else:
    token = getpass.getpass("Enter GitHub token (or press Enter to skip for public repo): ")

if token:
    !git clone https://{token}@github.com/Chalhotra/ViT-Token-Economy.git 2>&1 | grep -v 'http' || true
else:
    !git clone https://github.com/Chalhotra/ViT-Token-Economy.git

%cd ViT-Token-Economy

In [ ]:
!pip -q install -r requirements.txt
!pip -q install -e .

In [ ]:
from src.imagenet_mapping import build_imagenet100_to_1k_map
from src.models import ModelConfig, create_model, shrink_imagenet1k_head_to_imagenet100
from src.data import DataConfig, load_imagenet100_split, build_transform_for_model, apply_timm_preprocess, build_loader
from src.eval import evaluate_accuracy_latency_throughput, compute_gflops
from src.utils import get_device, num_params
import torch

In [ ]:
device = get_device()
maps = build_imagenet100_to_1k_map()

In [ ]:
def run(model_id: str, batch_size: int = 64):
    model = create_model(ModelConfig(model_id=model_id, pretrained=True))
    model = shrink_imagenet1k_head_to_imagenet100(model, maps.new_to_old_map, num_classes=100)
    model = model.to(device).eval()
    ds = load_imagenet100_split(DataConfig(split='validation'))
    transform = build_transform_for_model(model)
    ds_t = apply_timm_preprocess(ds, transform)
    loader = build_loader(ds_t, DataConfig(batch_size=batch_size, split='validation', shuffle=False))
    metrics = evaluate_accuracy_latency_throughput(model, loader, device)
    sample = ds_t[0]['pixel_values'].unsqueeze(0).to(device)
    gflops = compute_gflops(model, sample)
    return {
        'model': model_id,
        'params_m': num_params(model)/1e6,
        'gflops': gflops,
        **metrics
    }

In [ ]:
run('vit_tiny_patch16_224')

In [ ]:
run('deit_tiny_patch16_224')